# Customer Churn Prediction

## Phase 1: Data Understanding

### Business Problem

A telecommunications company wants to identify customers who are likely to
churn so that the retention team can proactively engage with them.

The objective of this project is to develop a machine learning solution that
predicts whether a customer is likely to churn based on their demographic,
account, service, and billing information.

### Dataset

IBM Telco Customer Churn Dataset

### Target Variable

`Churn`

- `Yes` = Customer churned
- `No` = Customer did not churn

This phase focuses on understanding the structure and quality of the dataset
before performing data cleaning, preprocessing, feature engineering, and
machine learning.

In [ ]:
%pip install pandas numpy matplotlib seaborn

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Visualization style
sns.set_style("whitegrid")

print("Libraries imported successfully.")


In [ ]:
file_path = "../data/WA_FnUseC_TelcoCustomerChurn.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")


In [ ]:
df_raw = df.copy()

print("Backup created.")
print("Original dataset shape:", df_raw.shape)


In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

df.shape


In [ ]:
df.head()


In [ ]:
df.tail()


In [ ]:
df.sample(5, random_state=42)


In [ ]:
print("Column names:")
for i, column in enumerate(df.columns, start=1):
    print(f"{i}. {column}")


In [ ]:
df.info()


In [ ]:
df.dtypes


In [ ]:
df.dtypes.value_counts()


In [ ]:
df.describe()


In [ ]:
df.describe(include="object")


In [ ]:
df.describe(include="all").T


In [ ]:
missing_values = df.isnull().sum()

print("Missing values by column:")
print(missing_values)


In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(
    by="Missing_Count",
    ascending=False
)

missing_summary


In [ ]:
blank_values = (df == "").sum()

print("Blank values by column:")
print(blank_values)

whitespace_values = df.apply(
    lambda column: column.astype(str).str.strip().eq("").sum()
)

whitespace_values


In [ ]:
print("TotalCharges data type:", df["TotalCharges"].dtype)
df["TotalCharges"].head(20)
df["TotalCharges"].tail(20)

total_charges_numeric = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

invalid_total_charges = total_charges_numeric.isna().sum()

print(
    "Values in TotalCharges that cannot be interpreted as numbers:",
    invalid_total_charges
)


In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)


In [ ]:
duplicate_percentage = (
    df.duplicated().mean() * 100
)

print(
    f"Duplicate row percentage: {duplicate_percentage:.2f}%"
)


In [ ]:
total_rows = len(df)
unique_customer_ids = df["customerID"].nunique()

print("Total rows:", total_rows)
print("Unique customer IDs:", unique_customer_ids)
print(
    "Duplicate customer IDs:",
    df["customerID"].duplicated().sum()
)


In [ ]:
numerical_features = df.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numerical features:")
for feature in numerical_features:
    print("-", feature)


In [ ]:
numerical_features = df.select_dtypes(
    include=np.number
).columns.tolist()

numerical_features


In [ ]:
categorical_features = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical/object features:")
for feature in categorical_features:
    print("-", feature)


In [ ]:
unique_summary = pd.DataFrame({
    "Column": df.columns,
    "Unique_Values": [df[col].nunique() for col in df.columns]
})

unique_summary


In [ ]:
for column in categorical_features:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")
    print(f"{'=' * 60}")
    print(df[column].unique())


In [ ]:
print("Gender:")
print(df["gender"].value_counts())

print("\nPartner:")
print(df["Partner"].value_counts())

print("\nDependents:")
print(df["Dependents"].value_counts())

print("\nInternet Service:")
print(df["InternetService"].value_counts())

print("\nContract:")
print(df["Contract"].value_counts())

print("\nPayment Method:")
print(df["PaymentMethod"].value_counts())


In [ ]:
print("Unique Churn values:")
print(df["Churn"].unique())


In [ ]:
churn_counts = df["Churn"].value_counts()

print(churn_counts)


In [ ]:
churn_percentages = df["Churn"].value_counts(
    normalize=True
) * 100

print(churn_percentages.round(2))


In [ ]:
churn_summary = pd.DataFrame({
    "Count": df["Churn"].value_counts(),
    "Percentage": df["Churn"].value_counts(normalize=True) * 100
})

churn_summary["Percentage"] = churn_summary["Percentage"].round(2)

churn_summary


In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="Churn",
    hue="Churn",
    palette=["#4CAF50", "#E53935"],
    legend=False
)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()


In [ ]:
print("Average tenure:", round(df["tenure"].mean(), 2))
print("Median tenure:", df["tenure"].median())

print(
    "Average monthly charges:",
    round(df["MonthlyCharges"].mean(), 2)
)

print(
    "Median monthly charges:",
    round(df["MonthlyCharges"].median(), 2)
)


In [ ]:
print(df["Contract"].value_counts())


In [ ]:
print(df["InternetService"].value_counts())


In [ ]:
print(df["PaymentMethod"].value_counts())


In [ ]:
pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
).round(3)


In [ ]:
pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
).round(3)


In [ ]:
pd.crosstab(
    df["gender"],
    df["Churn"],
    normalize="index"
).round(3)


In [ ]:
print("Potentially problematic columns:\n")

print("1. customerID")
print("   Reason: Identifier; should not be used as a predictive feature.")

print("\n2. TotalCharges")
print("   Reason: May be stored as object/string and may contain blank values.")

print("\n3. Churn")
print("   Reason: Target variable; should not be included as an input feature.")


## Initial Data Understanding – Key Observations

The dataset contains customer-level information covering demographics,
account information, subscribed services, contract details, billing
information, and churn status.

The target variable is `Churn`, which contains two classes: `Yes` and `No`.

The dataset contains both numerical and categorical variables. Important
numerical variables include `tenure` and `MonthlyCharges`. `TotalCharges`
represents a numerical business measure but may initially be stored as a
text/object data type.

`customerID` is a unique customer identifier. It does not represent a
customer characteristic and therefore should not be used as a predictive
feature in the machine learning model.

The missing-value analysis and blank-string analysis are important because
some values in `TotalCharges` may be represented as blank strings rather
than standard missing values.

Duplicate rows and duplicate customer IDs were also investigated.

The target variable distribution was examined to determine whether the
dataset has a class imbalance between customers who churned and customers
who remained.

At this stage, no rows or columns have been removed and no values have
been imputed or encoded. These preprocessing decisions will be performed
in the next phase using a reproducible preprocessing pipeline.


In [ ]:
data_dictionary = pd.DataFrame({
    "Feature": df.columns,
    "Data_Type": df.dtypes.astype(str),
    "Missing_Values": df.isnull().sum(),
    "Unique_Values": df.nunique()
})

data_dictionary

data_dictionary.to_csv(
    "../data/data_dictionary_summary.csv",
    index=False
)


In [ ]:
print("=" * 60)
print("PHASE 1 - DATA UNDERSTANDING SUMMARY")
print("=" * 60)

print(f"\nRows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

print(f"\nDuplicate rows: {df.duplicated().sum()}")

print(
    f"Unique customer IDs: {df['customerID'].nunique()}"
)

print("\nNumerical columns:")
print(numerical_features)

print("\nObject/Categorical columns:")
print(categorical_features)

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nChurn distribution:")
print(df["Churn"].value_counts())

print("\nChurn percentage:")
print(
    (df["Churn"].value_counts(normalize=True) * 100).round(2)
)

print("\nTotalCharges data type:")
print(df["TotalCharges"].dtype)

print(
    "\nInvalid TotalCharges values:",
    pd.to_numeric(
        df["TotalCharges"],
        errors="coerce"
    ).isna().sum()
)

print("\n" + "=" * 60)


# ============================================================
# PHASE 2 — DATA CLEANING & PREPROCESSING
# ============================================================


# Phase 2: Data Cleaning & Preprocessing

The objective of this phase is to prepare the dataset for machine learning
while preventing data leakage.

The preprocessing steps will be implemented using scikit-learn pipelines
so that the same transformations can be consistently applied to both the
test data and new/unseen customer data.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer


In [ ]:
file_path = "../data/WA_FnUseC_TelcoCustomerChurn.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

data = df.copy()

print("Working dataset shape:", data.shape)

data = data.drop(columns=["customerID"])

print("Shape after removing customerID:", data.shape)


In [ ]:
print(data["TotalCharges"].dtype)

data["TotalCharges"] = pd.to_numeric(
    data["TotalCharges"],
    errors="coerce"
)

print(data["TotalCharges"].dtype)

missing_values = data.isnull().sum()

missing_values[missing_values > 0]

print(
    "Missing TotalCharges:",
    data["TotalCharges"].isnull().sum()
)


In [ ]:
data["Churn"].value_counts()

data["Churn"] = data["Churn"].map({
    "No": 0,
    "Yes": 1
})

data["Churn"].value_counts()

print(data["Churn"].unique())

print(data["Churn"].isnull().sum())


In [ ]:
X = data.drop(columns=["Churn"])
y = data["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical features:")
for col in numerical_features:
    print("-", col)

print("\nCategorical features:")
for col in categorical_features:
    print("-", col)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)


In [ ]:
print("Overall churn distribution:")
print(y.value_counts(normalize=True).round(3))

print("\nTraining churn distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting churn distribution:")
print(y_test.value_counts(normalize=True).round(3))

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

for feature in feature_names:
    print(feature)


In [ ]:
X_train_processed_df = pd.DataFrame(
    X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()
X_train_processed_df.shape


In [ ]:
print(
    "Missing values after preprocessing:",
    X_train_processed_df.isnull().sum().sum()
)


In [ ]:
print("=" * 60)
print("PHASE 2 PREPROCESSING SUMMARY")
print("=" * 60)

print(f"\nOriginal dataset: {df.shape}")

print(f"After removing customerID: {data.shape}")

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

print(f"\nOriginal feature count: {X.shape[1]}")
print(f"Processed feature count: {X_train_processed.shape[1]}")

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nMissing values in processed training data:")
print(X_train_processed_df.isnull().sum().sum())

print("\n" + "=" * 60)


## Avoiding Data Leakage

Data leakage occurs when information from the test dataset is used during
the training or preprocessing process.

To prevent leakage, the dataset was first divided into training and testing
sets using a 70:30 split with `random_state=42`.

The preprocessing pipeline was then fitted only on the training data.

Numerical missing values are handled using the median calculated from the
training data, while categorical missing values are handled using the most
frequent category from the training data.

Categorical variables are one-hot encoded using a transformer fitted on the
training data.

The fitted preprocessing pipeline is then used to transform both the
training and testing datasets.

This approach ensures that information from the test dataset does not
influence the preprocessing decisions made during model training.

The same fitted preprocessing pipeline can later be saved together with
the machine learning model and reused when predicting churn for new
customers through the REST API.


## Preprocessing Decisions

### 1. Customer ID

`customerID` was removed because it is an identifier and does not represent
a meaningful customer characteristic. Including it could allow the model
to learn arbitrary patterns associated with customer identifiers.

### 2. TotalCharges

`TotalCharges` was converted from an object/string representation to a
numeric data type. Values that could not be converted were represented as
missing values and will be handled by the numerical preprocessing pipeline.

### 3. Target Encoding

The target variable `Churn` was converted from:

- `No` → 0
- `Yes` → 1

This allows the classification model to work with a numerical target.

### 4. Numerical Features

Missing numerical values are handled using median imputation.

### 5. Categorical Features

Missing categorical values are handled using the most frequent category,
followed by one-hot encoding.

`handle_unknown="ignore"` is used so that the preprocessing pipeline can
handle previously unseen categories when new customer data is submitted.

### 6. Train/Test Split

The data was split into 70% training and 30% testing data using
`random_state=42`. Stratification was used to preserve the target class
distribution across both datasets.

### 7. Leakage Prevention

The preprocessing transformations are fitted only on the training dataset.
The test dataset is transformed using the already-fitted preprocessing
pipeline and is not used to learn preprocessing parameters.


# Phase 3: Exploratory Data Analysis

Exploratory Data Analysis (EDA) is performed to understand customer
characteristics and identify patterns associated with customer churn.

The analysis covers:

1. Overall churn distribution
2. Customer tenure distribution
3. Monthly charges distribution
4. Churn by contract type
5. Churn by internet service
6. Churn by payment method
7. Churn by technical support availability
8. Relationship between tenure and monthly charges

The visualizations are intended to identify patterns and relationships
that may be useful for predicting customer churn.


In [ ]:
eda_data = data.copy()

eda_data["ChurnLabel"] = eda_data["Churn"].map({
    0: "No",
    1: "Yes"
})

eda_data.head()

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.countplot(
    data=eda_data,
    x="ChurnLabel",
    hue="ChurnLabel",
    palette=["#4CAF50", "#E53935"],
    legend=False
)

plt.title("Customer Churn Distribution", fontsize=16, fontweight="bold")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")

# Add counts on bars
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.show()


In [ ]:
churn_percentage = (
    eda_data["ChurnLabel"]
    .value_counts(normalize=True)
    .mul(100)
)

plt.figure(figsize=(6, 6))

plt.pie(
    churn_percentage,
    labels=churn_percentage.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#4CAF50", "#E53935"]
)

plt.title("Percentage of Customers by Churn Status")

plt.show()


### Business Insight

The churn distribution shows the proportion of customers who left the
telecommunications service compared with those who remained.

The two classes are not equally represented, indicating that churn is a
class-imbalanced classification problem. This is important when evaluating
the machine learning model because accuracy alone may not fully describe
the model's ability to identify customers who churn.

Therefore, precision, recall, and F1 score will also be considered during
model evaluation.


In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=eda_data,
    x="tenure",
    bins=30,
    kde=True,
    color="#1976D2"
)

plt.title("Customer Tenure Distribution", fontsize=16, fontweight="bold")
plt.xlabel("Tenure (Months)")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=eda_data,
    x="ChurnLabel",
    y="tenure",
    hue="ChurnLabel",
    palette=["#4CAF50", "#E53935"],
    legend=False
)

plt.title("Customer Tenure by Churn Status", fontsize=16, fontweight="bold")
plt.xlabel("Churn Status")
plt.ylabel("Tenure (Months)")

plt.tight_layout()
plt.show()


### Business Insight

Customer tenure provides an indication of how long customers have remained
with the company.

The comparison of tenure across churn groups helps determine whether
customers who leave tend to have shorter or longer relationships with the
company.

If churned customers generally have lower tenure, this suggests that the
early stages of the customer lifecycle may be an important period for
retention activities.


In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    data=eda_data,
    x="MonthlyCharges",
    bins=30,
    kde=True,
    color="#7B1FA2"
)

plt.title(
    "Distribution of Monthly Charges",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Monthly Charges")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=eda_data,
    x="ChurnLabel",
    y="MonthlyCharges",
    hue="ChurnLabel",
    palette=["#4CAF50", "#E53935"],
    legend=False
)

plt.title(
    "Monthly Charges by Churn Status",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Churn Status")
plt.ylabel("Monthly Charges")

plt.tight_layout()
plt.show()


### Business Insight

Monthly charges show the amount customers are billed each month.

Comparing monthly charges between churn groups helps determine whether
customers with higher monthly charges appear more likely to churn.

Differences between the distributions can help the business identify
whether pricing or higher monthly spending may be associated with customer
retention risk. However, this relationship should not be interpreted as
causal based on EDA alone.


In [ ]:
contract_churn = pd.crosstab(
    eda_data["Contract"],
    eda_data["ChurnLabel"],
    normalize="index"
) * 100

contract_churn

contract_churn.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["#4CAF50", "#E53935"]
)

plt.title(
    "Churn Rate by Contract Type",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Contract Type")
plt.ylabel("Percentage of Customers")
plt.legend(title="Churn")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Business Insight

Contract type shows a clear difference in the proportion of customers
who churn across contract categories.

Month-to-month customers can be compared with customers on longer-term
contracts to understand whether contract commitment is associated with
customer retention.

If month-to-month customers show a higher churn proportion, the retention
team could consider targeted engagement strategies for customers who have
not committed to longer-term contracts.


In [ ]:
internet_churn = pd.crosstab(
    eda_data["InternetService"],
    eda_data["ChurnLabel"],
    normalize="index"
) * 100

internet_churn

internet_churn.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["#4CAF50", "#E53935"]
)

plt.title(
    "Churn Rate by Internet Service",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Internet Service")
plt.ylabel("Percentage of Customers")
plt.legend(title="Churn")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Business Insight

Churn rates differ across internet service categories.

This analysis can help the business determine whether customers using
different internet technologies have different retention patterns.

A noticeable difference between service types may indicate that service
experience, pricing, customer expectations, or other factors associated
with the service type deserve further investigation.

In [ ]:
payment_churn = pd.crosstab(
    eda_data["PaymentMethod"],
    eda_data["ChurnLabel"],
    normalize="index"
) * 100

payment_churn

plt.figure(figsize=(12, 6))

sns.barplot(
    data=eda_data,
    x="PaymentMethod",
    y="Churn",
    hue="PaymentMethod",
    estimator="mean",
    errorbar=None,
    palette="viridis",
    legend=False
)

plt.title(
    "Average Churn Rate by Payment Method",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Payment Method")
plt.ylabel("Churn Rate")

plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.show()


In [ ]:
payment_churn_rate = (
    eda_data.groupby("PaymentMethod")["Churn"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

plt.figure(figsize=(11, 6))

ax = sns.barplot(
    x=payment_churn_rate.index,
    y=payment_churn_rate.values,
    hue=payment_churn_rate.index,
    palette="viridis",
    legend=False
)

plt.title(
    "Churn Rate by Payment Method",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Payment Method")
plt.ylabel("Churn Rate (%)")

plt.xticks(rotation=30, ha="right")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()


In [ ]:
tech_support_rate = (
    eda_data.groupby("TechSupport")["Churn"]
    .mean()
    .sort_values(ascending=False)
    * 100
)

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=tech_support_rate.index,
    y=tech_support_rate.values,
    hue=tech_support_rate.index,
    palette="Set2",
    legend=False
)

plt.title(
    "Churn Rate by Technical Support",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Technical Support")
plt.ylabel("Churn Rate (%)")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=eda_data,
    x="tenure",
    y="MonthlyCharges",
    hue="ChurnLabel",
    palette={
        "No": "#4CAF50",
        "Yes": "#E53935"
    },
    alpha=0.6
)

plt.title(
    "Tenure vs Monthly Charges by Churn Status",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Tenure (Months)")
plt.ylabel("Monthly Charges")

plt.legend(title="Churn")

plt.tight_layout()
plt.show()


In [ ]:
senior_churn = (
    eda_data.groupby("SeniorCitizen")["Churn"]
    .mean()
    * 100
)

senior_labels = {
    0: "Non-Senior Citizen",
    1: "Senior Citizen"
}

senior_churn.index = senior_churn.index.map(senior_labels)

plt.figure(figsize=(8, 5))

ax = sns.barplot(
    x=senior_churn.index,
    y=senior_churn.values,
    hue=senior_churn.index,
    palette=["#42A5F5", "#FF7043"],
    legend=False
)

plt.title(
    "Churn Rate by Senior Citizen Status",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Customer Group")
plt.ylabel("Churn Rate (%)")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()


In [ ]:
numerical_eda = eda_data[
    [
        "SeniorCitizen",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "Churn"
    ]
]

correlation_matrix = numerical_eda.corr()

correlation_matrix

plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)

plt.title(
    "Correlation Matrix of Numerical Variables",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()


### Business Insight

The correlation matrix summarizes the linear relationships between the
main numerical variables.

Tenure and TotalCharges are expected to have a relationship because
TotalCharges accumulates over the customer's time with the company.

The correlation between individual numerical variables and churn can
provide an initial indication of potentially useful predictors. However,
correlation measures only certain types of relationships and should not be
used alone to determine feature importance.


In [ ]:
eda_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Churned Customers",
        "Non-Churned Customers",
        "Overall Churn Rate",
        "Average Tenure",
        "Average Monthly Charges"
    ],
    "Value": [
        len(eda_data),
        (eda_data["Churn"] == 1).sum(),
        (eda_data["Churn"] == 0).sum(),
        f"{eda_data['Churn'].mean() * 100:.2f}%",
        f"{eda_data['tenure'].mean():.2f} months",
        f"{eda_data['MonthlyCharges'].mean():.2f}"
    ]
})

eda_summary


## Overall EDA Findings

The exploratory analysis identified several customer characteristics that
show differences in churn behaviour.

### Key observations

1. The churn target is imbalanced, with fewer customers churning than
   remaining with the company.

2. Customer tenure shows differences between churned and non-churned
   customers, indicating that customer lifecycle stage may be relevant to
   churn prediction.

3. Monthly charges show differences between churn groups, suggesting that
   billing level may contain predictive information.

4. Contract type shows noticeable differences in churn rates. Contract
   structure therefore appears to be an important customer characteristic
   for further modelling.

5. Churn rates vary across internet service categories, indicating that
   service type may contain useful predictive information.

6. Payment method also shows differences in churn rates across customer
   groups.

7. Technical support availability is associated with differences in churn
   rates and may therefore provide useful predictive information.

8. The relationship between tenure and monthly charges suggests that
   multiple customer characteristics may need to be considered together
   rather than examining each variable independently.

These findings are exploratory and describe associations in the dataset.
They do not establish causal relationships.

The observations will be considered during feature engineering and model
development. The Decision Tree model in the next phases will determine
which features provide useful predictive information when considered
together.

# Phase 4: Feature Engineering

Feature engineering involves creating new variables from existing customer
information that may provide additional predictive information to the
machine learning model.

Three features are created:

1. AverageMonthlySpend
2. TotalServices
3. TenureGroup

The engineered features are created using information available for each
customer and do not use the target variable `Churn`.


In [ ]:
# Create a copy of the cleaned dataset
feature_data = data.copy()

print("Dataset shape before feature engineering:", feature_data.shape)


In [ ]:
feature_data["AverageMonthlySpend"] = np.where(
    feature_data["tenure"] > 0,
    feature_data["TotalCharges"] / feature_data["tenure"],
    feature_data["MonthlyCharges"]
)


feature_data[
    [
        "tenure",
        "TotalCharges",
        "MonthlyCharges",
        "AverageMonthlySpend"
    ]
].head(10)

feature_data["AverageMonthlySpend"].describe()


In [ ]:
service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in service_columns:
    print(col, ":", col in feature_data.columns)

feature_data["TotalServices"] = (
    feature_data[service_columns]
    .eq("Yes")
    .sum(axis=1)
)

feature_data[
    service_columns + ["TotalServices"]
].head(10)
feature_data["TotalServices"].describe()


In [ ]:
service_churn = (
    feature_data.groupby("TotalServices")["Churn"]
    .mean()
    .mul(100)
)

print(service_churn)

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=service_churn.index,
    y=service_churn.values,
    color="#7B1FA2"
)

plt.title(
    "Churn Rate by Number of Active Services",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Number of Active Services")
plt.ylabel("Churn Rate (%)")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()


### TotalServices

`TotalServices` represents the number of active telecom services associated
with each customer.

A service is counted when its corresponding column contains `Yes`.

This feature provides a summary measure of customer engagement with the
company. Customers with different numbers of subscribed services may have
different churn behaviour.

The feature does not use `Churn` to calculate the value, so target leakage
is avoided.


In [ ]:
def create_tenure_group(tenure):
    if tenure <= 12:
        return "New"
    elif tenure <= 24:
        return "Developing"
    elif tenure <= 48:
        return "Established"
    else:
        return "Loyal"

feature_data["TenureGroup"] = feature_data["tenure"].apply(
    create_tenure_group
)

feature_data[
    ["tenure", "TenureGroup"]
].head(20)


In [ ]:
tenure_order = [
    "New",
    "Developing",
    "Established",
    "Loyal"
]

feature_data["TenureGroup"] = pd.Categorical(
    feature_data["TenureGroup"],
    categories=tenure_order,
    ordered=True
)

feature_data["TenureGroup"].value_counts().sort_index()


In [ ]:
tenure_churn = (
    feature_data
    .groupby("TenureGroup", observed=True)["Churn"]
    .mean()
    .mul(100)
)

print(tenure_churn)

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=tenure_churn.index,
    y=tenure_churn.values,
    color="#00897B"
)

plt.title(
    "Churn Rate by Tenure Group",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()


### TenureGroup

`TenureGroup` categorizes customers according to the length of their
relationship with the company.

The groups are:

- New: 0–12 months
- Developing: 13–24 months
- Established: 25–48 months
- Loyal: 49+ months

This feature may help the model identify differences in churn behaviour
across different stages of the customer lifecycle.

The feature is derived only from `tenure` and does not use the target
variable.


In [ ]:
engineered_features = [
    "AverageMonthlySpend",
    "TotalServices",
    "TenureGroup"
]

feature_data[engineered_features].head(10)

feature_data[engineered_features].isnull().sum()

feature_data[
    [
        "AverageMonthlySpend",
        "TotalServices"
    ]
].describe()


In [ ]:
X = feature_data.drop(columns=["Churn"])
y = feature_data["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


In [ ]:
numerical_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)


In [ ]:
numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            numerical_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


In [ ]:
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)


In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

for feature in feature_names:
    print(feature)


## Phase 4 Summary

Three features were engineered:

### AverageMonthlySpend

Calculated using:

`TotalCharges / tenure`

with `MonthlyCharges` used for customers with zero tenure.

This provides a normalized representation of customer spending.

### TotalServices

Calculated by counting the number of active telecom services for each
customer.

This provides a summary measure of customer engagement with the company.

### TenureGroup

Customers were grouped into:

- New: 0–12 months
- Developing: 13–24 months
- Established: 25–48 months
- Loyal: 49+ months

This represents different stages of the customer lifecycle.

All engineered features are derived from information available for the
customer and do not use the `Churn` target.

The train/test split was recreated after feature engineering using a
70:30 split, `random_state=42`, and stratification.

The preprocessing pipeline was also rebuilt and fitted only on the
training data to prevent data leakage.


# Phase 5: Decision Tree Model Development

A Decision Tree Classifier is used to predict whether a customer will churn.

Two different Decision Tree configurations are trained and compared.

The models use the preprocessing pipeline developed in the previous phase.
This ensures that numerical and categorical variables are transformed
consistently.

The comparison focuses on the models' predictive performance while also
considering model complexity and interpretability.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)


In [ ]:
dt_model_1 = DecisionTreeClassifier(
    random_state=42
)

dt_model_1.fit(
    X_train_processed,
    y_train
)


In [ ]:
y_pred_1 = dt_model_1.predict(X_test_processed)

y_prob_1 = dt_model_1.predict_proba(
    X_test_processed
)[:, 1]

print("Model 1 depth:", dt_model_1.get_depth())
print("Model 1 leaves:", dt_model_1.get_n_leaves())

print(
    "Model 1 number of nodes:",
    dt_model_1.tree_.node_count
)



In [ ]:
dt_model_2 = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

dt_model_2.fit(
    X_train_processed,
    y_train
)


y_pred_2 = dt_model_2.predict(
    X_test_processed
)

y_prob_2 = dt_model_2.predict_proba(
    X_test_processed
)[:, 1]


In [ ]:
print("Model 2 depth:", dt_model_2.get_depth())
print("Model 2 leaves:", dt_model_2.get_n_leaves())

print(
    "Model 2 number of nodes:",
    dt_model_2.tree_.node_count
)


In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0)
    }

metrics_1 = calculate_metrics(
    y_test,
    y_pred_1
)

metrics_2 = calculate_metrics(
    y_test,
    y_pred_2
)

comparison = pd.DataFrame(
    [metrics_1, metrics_2],
    index=[
        "Decision Tree - Baseline",
        "Decision Tree - Controlled"
    ]
)

comparison

comparison.round(4)


In [ ]:
comparison["Tree Depth"] = [
    dt_model_1.get_depth(),
    dt_model_2.get_depth()
]

comparison["Number of Leaves"] = [
    dt_model_1.get_n_leaves(),
    dt_model_2.get_n_leaves()
]

comparison.round(4)


In [ ]:
metrics_to_plot = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

comparison[metrics_to_plot].plot(
    kind="bar",
    figsize=(10, 6),
    color=[
        "#1976D2",
        "#7B1FA2",
        "#00897B",
        "#F57C00"
    ]
)

plt.title(
    "Decision Tree Model Comparison",
    fontsize=16,
    fontweight="bold"
)

plt.ylabel("Score")
plt.xlabel("Model")

plt.ylim(0, 1)

plt.xticks(rotation=0)

plt.legend(
    title="Metric",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


In [ ]:
train_pred_1 = dt_model_1.predict(
    X_train_processed
)

test_pred_1 = dt_model_1.predict(
    X_test_processed
)

print(
    "Model 1 Training Accuracy:",
    accuracy_score(y_train, train_pred_1)
)

print(
    "Model 1 Testing Accuracy:",
    accuracy_score(y_test, test_pred_1)
)

train_pred_2 = dt_model_2.predict(
    X_train_processed
)

test_pred_2 = dt_model_2.predict(
    X_test_processed
)

print(
    "Model 2 Training Accuracy:",
    accuracy_score(y_train, train_pred_2)
)

print(
    "Model 2 Testing Accuracy:",
    accuracy_score(y_test, test_pred_2)
)


## Training vs Testing Performance

Training performance indicates how well the model fits the data it learned
from, while testing performance indicates how well it generalizes to
unseen data.

A large difference between training and testing performance can indicate
overfitting.

The baseline Decision Tree is allowed to grow without an explicit depth
restriction and may therefore produce a more complex tree.

The controlled Decision Tree restricts the tree depth and minimum split
size, which reduces model complexity and can improve generalization.

The final model will be selected based on the observed test-set metrics,
with particular attention to recall and F1 score because the business
objective involves identifying customers who may churn.


In [ ]:
model_summary = pd.DataFrame({
    "Model": [
        "Baseline Decision Tree",
        "Controlled Decision Tree"
    ],
    "Max Depth": [
        "None",
        5
    ],
    "Min Samples Split": [
        2,
        20
    ],
    "Accuracy": [
        metrics_1["Accuracy"],
        metrics_2["Accuracy"]
    ],
    "Precision": [
        metrics_1["Precision"],
        metrics_2["Precision"]
    ],
    "Recall": [
        metrics_1["Recall"],
        metrics_2["Recall"]
    ],
    "F1 Score": [
        metrics_1["F1 Score"],
        metrics_2["F1 Score"]
    ],
    "Tree Depth": [
        dt_model_1.get_depth(),
        dt_model_2.get_depth()
    ],
    "Leaves": [
        dt_model_1.get_n_leaves(),
        dt_model_2.get_n_leaves()
    ]
})

model_summary.round(4)

if metrics_2["F1 Score"] >= metrics_1["F1 Score"]:
    final_model = dt_model_2
    final_model_name = "Controlled Decision Tree"
else:
    final_model = dt_model_1
    final_model_name = "Baseline Decision Tree"

print("Selected model:", final_model_name)


In [ ]:
print("Final model:")
print(final_model)

print("Final model depth:", final_model.get_depth())
print("Final model leaves:", final_model.get_n_leaves())


## Model Development Summary

Two Decision Tree configurations were developed.

### Model 1: Baseline Decision Tree

The baseline model uses the default Decision Tree configuration with
`random_state=42`.

This allows the tree to grow without an explicit maximum depth restriction.
It provides a useful baseline for comparison but may produce a relatively
complex tree.

### Model 2: Controlled Decision Tree

The second model uses:

- `max_depth=5`
- `min_samples_split=20`
- `random_state=42`

These parameters restrict tree complexity and can help reduce overfitting.

### Model Comparison

The two models were compared using:

- Accuracy
- Precision
- Recall
- F1 Score

Tree depth and number of leaves were also examined to understand model
complexity.

The final model was selected based primarily on F1 Score while considering
recall and model complexity.

The exact model selected and its performance are reported using the
metrics generated in the notebook.


# Phase 6: Model Evaluation

The selected Decision Tree model is evaluated on the unseen test dataset.

The evaluation uses:

- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix

The test dataset was not used during model training or preprocessing
fitting. Therefore, these metrics provide an estimate of how the selected
model performs on previously unseen customer data.


In [ ]:
print("Final model:", final_model_name)
y_pred = final_model.predict(X_test_processed)

y_pred_probability = final_model.predict_proba(
    X_test_processed
)[:, 1]
print(y_pred[:10])
print(y_pred_probability[:10])


In [ ]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")


In [ ]:
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

print(f"Precision: {precision:.4f}")
print(f"Precision: {precision * 100:.2f}%")


In [ ]:
recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

print(f"Recall: {recall:.4f}")
print(f"Recall: {recall * 100:.2f}%")


In [ ]:
f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

print(f"F1 Score: {f1:.4f}")
print(f"F1 Score: {f1 * 100:.2f}%")


In [ ]:
final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

final_metrics["Percentage"] = (
    final_metrics["Score"] * 100
).round(2)

final_metrics.round(4)


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    )
)


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)


In [ ]:
plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title(
    "Confusion Matrix - Final Decision Tree",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.tight_layout()
plt.show()


In [ ]:
tn, fp, fn, tp = cm.ravel()

print("True Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)


In [ ]:
cm_percentage = (
    cm / cm.sum()
) * 100

cm_percentage

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm_percentage,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title(
    "Confusion Matrix (%)",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.tight_layout()
plt.show()


## Business Interpretation of the Confusion Matrix

The confusion matrix contains four possible outcomes.

### True Negative (TN)

A customer who did not churn and was correctly predicted as not churning.

No unnecessary retention intervention is triggered.

### True Positive (TP)

A customer who actually churned and was correctly identified by the model.

This represents a customer for whom a proactive retention intervention
could potentially have been considered.

### False Positive (FP)

A customer who did not churn but was predicted as likely to churn.

The retention team may spend resources contacting a customer who would have
remained with the company.

### False Negative (FN)

A customer who actually churned but was predicted as not churning.

This represents a missed opportunity to identify a customer before churn.

For a churn prediction system, false negatives can be particularly
important because the company may lose the opportunity to intervene before
the customer leaves.


In [ ]:
false_negative_rate = fn / (fn + tp)

print(
    f"False Negative Rate: {false_negative_rate:.4f}"
)

print(
    f"False Negative Rate: {false_negative_rate * 100:.2f}%"
)

false_positive_rate = fp / (fp + tn)

print(
    f"False Positive Rate: {false_positive_rate:.4f}"
)

print(
    f"False Positive Rate: {false_positive_rate * 100:.2f}%"
)


In [ ]:
business_summary = pd.DataFrame({
    "Measure": [
        "Total Test Customers",
        "Actual Churners",
        "Actual Non-Churners",
        "Correctly Identified Churners",
        "Missed Churners",
        "Customers Incorrectly Flagged",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value": [
        len(y_test),
        int(y_test.sum()),
        int((y_test == 0).sum()),
        int(tp),
        int(fn),
        int(fp),
        f"{accuracy * 100:.2f}%",
        f"{precision * 100:.2f}%",
        f"{recall * 100:.2f}%",
        f"{f1 * 100:.2f}%"
    ]
})

business_summary


In [ ]:
metric_names = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

metric_values = [
    accuracy,
    precision,
    recall,
    f1
]

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=metric_names,
    y=metric_values,
    color="#1976D2"
)

plt.title(
    "Final Decision Tree Performance",
    fontsize=16,
    fontweight="bold"
)

plt.ylabel("Score")
plt.xlabel("Metric")
plt.ylim(0, 1)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.2f",
        padding=3
    )

plt.tight_layout()
plt.show()


## Final Model Evaluation

The selected Decision Tree model was evaluated using the unseen test
dataset.

The model achieved the following performance:

- Accuracy: [INSERT VALUE FROM OUTPUT]
- Precision: [INSERT VALUE FROM OUTPUT]
- Recall: [INSERT VALUE FROM OUTPUT]
- F1 Score: [INSERT VALUE FROM OUTPUT]

The confusion matrix provides additional insight into the types of
prediction errors made by the model.

The model correctly identified `TP` actual churners, while `FN` churners
were incorrectly classified as non-churners. It also produced `FP` false
positive predictions, where customers who did not churn were classified as
potential churners.

From a business perspective, false negatives are important because they
represent customers who eventually churn but were not identified by the
model. These customers may represent missed opportunities for proactive
retention.

False positives also have a business cost because they may result in
retention resources being spent on customers who would not have churned.

For this reason, recall is an important metric for the churn prediction
problem. However, precision must also be considered because a model with
very low precision could generate an excessive number of unnecessary
retention interventions.

Overall, the selected model's performance should therefore be assessed
not only through accuracy but through the combined behaviour of precision,
recall, F1 score, and the confusion matrix.


# Phase 7: Model Interpretation

The Decision Tree model is interpretable because its predictions are based
on a sequence of feature-based decisions.

In this phase, we examine:

1. Feature importance
2. The most influential features
3. Decision Tree structure
4. Business interpretation of the model

Feature importance is calculated from the trained Decision Tree and is used
to identify which input variables contributed most to the model's decisions.


In [ ]:
print("Final model:", final_model_name)
print("Number of input features:", len(feature_names))

print("Tree depth:", final_model.get_depth())
print("Number of leaves:", final_model.get_n_leaves())


In [ ]:
importances = final_model.feature_importances_

print("Number of importance values:", len(importances))
print("Number of feature names:", len(feature_names))


In [ ]:
feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

feature_importance_df = (
    feature_importance_df
    .sort_values(
        by="Importance",
        ascending=False
    )
    .reset_index(drop=True)
)
feature_importance_df.head(20)



In [ ]:
top_10_features = feature_importance_df.head(10)

top_10_features
top_10_features.style.format({
    "Importance": "{:.4f}"
})



In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=top_10_features,
    x="Importance",
    y="Feature",
    color="#1976D2"
)

plt.title(
    "Top 10 Features Influencing Churn Predictions",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()


In [ ]:
top_10_features = top_10_features.copy()

top_10_features["Importance_Percentage"] = (
    top_10_features["Importance"] * 100
)

top_10_features.style.format({
    "Importance": "{:.4f}",
    "Importance_Percentage": "{:.2f}%"
})


In [ ]:
feature_importance_df["CumulativeImportance"] = (
    feature_importance_df["Importance"].cumsum()
)

feature_importance_df.head(10)

top_10_cumulative = (
    feature_importance_df
    .head(10)["Importance"]
    .sum()
)

print(
    f"Top 10 features account for approximately "
    f"{top_10_cumulative * 100:.2f}% of total feature importance."
)


## Important Note About Feature Importance

Decision Tree feature importance measures how much each feature contributes
to reducing impurity across the tree.

A high feature importance means that the feature was useful for making
splits in this particular trained tree.

Feature importance should not be interpreted as proof that a feature
causes churn.

It also does not necessarily indicate whether the feature increases or
decreases churn. It only indicates its contribution to the model's
predictions.

Therefore, feature importance describes the model's behaviour rather than
establishing causal relationships.


In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(24, 12))

plot_tree(
    final_model,
    feature_names=feature_names,
    class_names=["No Churn", "Churn"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9
)

plt.title(
    "Decision Tree Structure (First 3 Levels)",
    fontsize=18,
    fontweight="bold"
)

plt.show()


In [ ]:
plt.figure(figsize=(30, 18))

plot_tree(
    final_model,
    feature_names=feature_names,
    class_names=["No Churn", "Churn"],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title(
    "Final Decision Tree",
    fontsize=20,
    fontweight="bold"
)

plt.show()


In [ ]:
plt.figure(figsize=(24, 12))

plot_tree(
    final_model,
    feature_names=feature_names,
    class_names=["No Churn", "Churn"],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=9
)

plt.title(
    "Decision Tree Structure - First 3 Levels",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "../model/decision_tree.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
top_features = feature_importance_df.head(10)["Feature"].tolist()

for i, feature in enumerate(top_features, start=1):
    print(f"{i}. {feature}")

def clean_feature_name(feature):
    feature = feature.replace("num__", "")
    feature = feature.replace("cat__", "")
    return feature
feature_importance_df["CleanFeature"] = (
    feature_importance_df["Feature"]
    .apply(clean_feature_name)
)

feature_importance_df.head(10)[
    ["CleanFeature", "Importance"]
]


In [ ]:
top_10_clean = (
    feature_importance_df
    .head(10)
    .sort_values(
        "Importance",
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_clean["CleanFeature"],
    top_10_clean["Importance"],
    color="#1976D2"
)

plt.title(
    "Top 10 Features Influencing Churn",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()


In [ ]:
engineered_importance = feature_importance_df[
    feature_importance_df["Feature"].str.contains(
        "AverageMonthlySpend|TotalServices|TenureGroup",
        case=False,
        regex=True
    )
]

engineered_importance

zero_importance = feature_importance_df[
    feature_importance_df["Importance"] == 0
]

print(
    "Number of features with zero importance:",
    len(zero_importance)
)

zero_importance.head(20)


In [ ]:
report_feature_importance = (
    feature_importance_df
    .head(10)
    [["CleanFeature", "Importance"]]
    .copy()
)

report_feature_importance["Importance (%)"] = (
    report_feature_importance["Importance"] * 100
).round(2)

report_feature_importance = (
    report_feature_importance
    .drop(columns=["Importance"])
)

report_feature_importance


# Phase 7 Summary

The final Decision Tree was interpreted using feature importance and a
visual representation of the tree structure.

Feature importance was extracted from the trained Decision Tree and the
top 10 features were identified.

The feature importance analysis helps explain which customer attributes
the model relied on most when predicting churn.

The Decision Tree visualization provides an additional interpretation of
the model by showing the sequence of decisions used to classify customers.

The analysis also examined the importance of the engineered features:

- AverageMonthlySpend
- TotalServices
- TenureGroup

Feature importance describes the contribution of variables to the trained
model's predictions and should not be interpreted as evidence of a causal
relationship with churn.

The main business takeaway is that the model can provide the retention team
with information about which customer characteristics are most useful for
identifying potential churners.


## Creating a Reusable Feature Engineering Transformer

The model requires three engineered features:

- AverageMonthlySpend
- TotalServices
- TenureGroup

To ensure the same feature engineering is applied to both training data
and future API requests, the feature engineering logic is encapsulated in
a reusable scikit-learn transformer.

This prevents inconsistencies between the training workflow and production
predictions.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# class ChurnFeatureEngineer(BaseEstimator, TransformerMixin):

#     def __init__(self):
#         self.service_columns = [
#             "PhoneService",
#             "MultipleLines",
#             "OnlineSecurity",
#             "OnlineBackup",
#             "DeviceProtection",
#             "TechSupport",
#             "StreamingTV",
#             "StreamingMovies"
#         ]

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         X = X.copy()

#         # Average monthly spend
#         X["AverageMonthlySpend"] = np.where(
#             X["tenure"] > 0,
#             X["TotalCharges"] / X["tenure"],
#             X["MonthlyCharges"]
#         )

#         # Total active services
#         X["TotalServices"] = (
#             X[self.service_columns]
#             .eq("Yes")
#             .sum(axis=1)
#         )

#         # Tenure group
#         X["TenureGroup"] = X["tenure"].apply(
#             self.create_tenure_group
#         )

#         return X

#     @staticmethod
#     def create_tenure_group(tenure):

#         if tenure <= 12:
#             return "New"

#         elif tenure <= 24:
#             return "Developing"

#         elif tenure <= 48:
#             return "Established"

#         else:
#             return "Loyal"

import sys
import os

sys.path.append(
    os.path.abspath("../")
)
from feature_engineering import ChurnFeatureEngineer


In [ ]:
feature_engineer = ChurnFeatureEngineer()

test_features = feature_engineer.transform(
    X_test.head()
)

test_features[
    [
        "tenure",
        "TotalCharges",
        "AverageMonthlySpend",
        "TotalServices",
        "TenureGroup"
    ]
]


In [ ]:
raw_X = data.drop(columns=["Churn"])
raw_y = data["Churn"]

print(raw_X.shape)
print(raw_y.shape)


In [ ]:
raw_X_train, raw_X_test, raw_y_train, raw_y_test = train_test_split(
    raw_X,
    raw_y,
    test_size=0.30,
    random_state=42,
    stratify=raw_y
)

print("Raw training:", raw_X_train.shape)
print("Raw testing:", raw_X_test.shape)


In [ ]:
raw_numerical_features = raw_X_train.select_dtypes(
    include=np.number
).columns.tolist()

raw_categorical_features = raw_X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical:")
print(raw_numerical_features)

print("\nCategorical:")
print(raw_categorical_features)


In [ ]:
pipeline_numerical_features = raw_numerical_features + [
    "AverageMonthlySpend",
    "TotalServices"
]

pipeline_categorical_features = raw_categorical_features + [
    "TenureGroup"
]

print("Pipeline numerical features:")
print(pipeline_numerical_features)

print("\nPipeline categorical features:")
print(pipeline_categorical_features)


In [ ]:
pipeline_numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)

pipeline_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


In [ ]:
pipeline_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            pipeline_numerical_transformer,
            pipeline_numerical_features
        ),
        (
            "cat",
            pipeline_categorical_transformer,
            pipeline_categorical_features
        )
    ]
)


In [ ]:
from sklearn.pipeline import Pipeline

final_pipeline = Pipeline(
    steps=[
        (
            "feature_engineering",
            ChurnFeatureEngineer()
        ),
        (
            "preprocessing",
            pipeline_preprocessor
        ),
        (
            "classifier",
            final_model
        )
    ]
)


In [ ]:
final_pipeline.fit(
    raw_X_train,
    raw_y_train
)


In [ ]:
pipeline_predictions = final_pipeline.predict(
    raw_X_test
)

pipeline_probabilities = final_pipeline.predict_proba(
    raw_X_test
)[:, 1]

pipeline_accuracy = accuracy_score(
    raw_y_test,
    pipeline_predictions
)

pipeline_precision = precision_score(
    raw_y_test,
    pipeline_predictions,
    zero_division=0
)

pipeline_recall = recall_score(
    raw_y_test,
    pipeline_predictions,
    zero_division=0
)

pipeline_f1 = f1_score(
    raw_y_test,
    pipeline_predictions,
    zero_division=0
)

print(f"Accuracy : {pipeline_accuracy:.4f}")
print(f"Precision: {pipeline_precision:.4f}")
print(f"Recall   : {pipeline_recall:.4f}")
print(f"F1 Score : {pipeline_f1:.4f}")


In [ ]:
single_customer = raw_X_test.iloc[[0]]

prediction = final_pipeline.predict(
    single_customer
)

probability = final_pipeline.predict_proba(
    single_customer
)[:, 1]

print("Prediction:", prediction[0])
print("Probability:", probability[0])


In [ ]:
import joblib

joblib.dump(
    final_pipeline,
    "../model/churn_model.pkl"
)


In [ ]:
loaded_pipeline = joblib.load(
    "../model/churn_model.pkl"
)


In [ ]:
loaded_prediction = loaded_pipeline.predict(
    single_customer
)

loaded_probability = loaded_pipeline.predict_proba(
    single_customer
)[:, 1]

print("Prediction:", loaded_prediction[0])
print("Probability:", loaded_probability[0])


In [ ]:
print(
    "Prediction matches:",
    prediction[0] == loaded_prediction[0]
)
